Assignment for Day 2

1. Load ES futures data (from folder market_data: ES[M][yy].csv)
2. Apply futures roll method to adjust ESZ25
3. Load SPY data from yf (fallback: SPX.csv)
4. Visualize histogram of daily returns
5. Calculate skewness and excess kurtosis

In [33]:
import pandas as pd
import numpy as np
import plotly.express as px
import yfinance as yf
from scipy import stats

TICKERS = ['ESM25', 'ESU25', 'ESZ25', 'ESH26', 'ESM26', 'ESU26']


In [2]:
data = {}
close_prices = []
volumes = []
# load futures data in folder market_data
for ticker in TICKERS:
    print(f"Loading data for {ticker}")
    try:
        data[ticker] = pd.read_csv(f"../../market_data/{ticker}.csv")
        data[ticker].set_index("Time", inplace=True)
        data[ticker].index = pd.to_datetime(data[ticker].index)
        # extract close prices and concatenate
        dfc = data[ticker][['Latest']]
        dfc.columns = [ticker]
        close_prices.append(dfc)
        # extract volumes and concatenate
        dfv = data[ticker][['Volume']]
        dfv.columns = [ticker]
        volumes.append(dfv)
    except FileNotFoundError:
        print(f"File not found for {ticker}")

Loading data for ESM25
Loading data for ESU25
Loading data for ESZ25
Loading data for ESH26
Loading data for ESM26
Loading data for ESU26


In [3]:
data[TICKERS[0]].head()

,Open,High,Low,Latest,Change,%Change,Volume,Open Int
Time,,,,,,,,
2025-06-20,5984.00,6019.25,5918.25,6010.21,28.71,0.48%,71380,0
2025-06-18,5977.25,6020.75,5964.75,5981.50,-3.50,-0.06%,453191,562716
2025-06-17,6038.50,6040.50,5976.75,5985.00,-50.75,-0.84%,870571,667682
2025-06-16,5949.00,6055.25,5944.00,6035.75,56.50,0.94%,1744867,1017789
2025-06-13,6045.00,6045.00,5927.50,5979.25,-70.25,-1.16%,2162167,1664937


In [4]:
df_close = pd.concat(close_prices, axis=1)
df_close.head()


/var/folders/82/hh8lkwfx2s5bvc9lnq2rp_0w0000gn/T/ipykernel_29094/1232281780.py:1: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df_close = pd.concat(close_prices, axis=1)


,ESM25,ESU25,ESZ25,ESH26,ESM26,ESU26
Time,,,,,,
2025-03-20,5712.75,NaN,NaN,NaN,NaN,NaN
2025-03-21,5718.25,NaN,NaN,NaN,NaN,NaN
2025-03-24,5815.50,NaN,NaN,NaN,NaN,NaN
2025-03-25,5826.50,NaN,NaN,NaN,NaN,NaN
2025-03-26,5759.50,NaN,NaN,NaN,NaN,NaN


In [ ]:
# plot unadjusted close prices using plotly
fig = px.line(df_close, y=df_close.columns, title='Unadjusted Close Prices')
fig.update_layout(xaxis_title=None, yaxis_title="Price")
fig.update_layout(legend_title='Contract')
fig.show()


In [70]:
# adjust the prices of ESZ25 by rolling to ESH26 on 12-11-2025
# load ESH26 data
df = df_close[['ESZ25', 'ESH26']]
df.head()

# create a new column for the adjusted prices
roll_date = pd.to_datetime('2025-12-11')
roll_gap = df.loc[roll_date, 'ESH26'] - df.loc[roll_date, 'ESZ25']
df['ESZ25_adjusted'] = df['ESZ25'] + roll_gap
df.loc[roll_date:, 'ESZ25_adjusted'] = np.nan

# plot the adjusted prices
fig = px.line(df, y=['ESZ25', 'ESZ25_adjusted', 'ESH26'], title='Adjusted Prices')
fig.update_layout(xaxis_title=None, yaxis_title="Price")
fig.update_layout(legend_title='Contract')
fig.show()


In [67]:
# plot volumes as bar charts
df_volumes = pd.concat(volumes, axis=1)

fig = px.bar(df_volumes, y=df_volumes.columns, title='Volumes by ES Futures Contract')
fig.update_layout(xaxis_title=None, yaxis_title="Volume")
fig.update_layout(legend_title_text='Contract')
fig.show()



/var/folders/82/hh8lkwfx2s5bvc9lnq2rp_0w0000gn/T/ipykernel_29094/3847511861.py:2: Pandas4Warning:

Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.



In [52]:
# load SPY prices
spy = yf.download("SPY", start="2023-01-01", end="2026-03-03", auto_adjust=True)
# drop 'Ticker' level from columns
spy = spy.droplevel(1, axis=1)
spy.head()

/opt/miniconda3/envs/algotrading/lib/python3.11/site-packages/yfinance/scrapers/history.py:172: Pandas4Warning:

Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.

[*********************100%***********************]  1 of 1 completed

Price,Close,High,Low,Open,Volume
Date,,,,,
2023-01-03,366.069122,371.461808,363.194919,369.481603,74850700
2023-01-04,368.895233,370.933111,365.280866,368.337683,85934100
2023-01-05,364.684875,367.049580,364.088896,366.934233,76970500
2023-01-06,373.047852,374.172545,364.713697,367.789729,104189600
2023-01-09,372.836365,378.450180,372.653751,375.249150,73978100


In [53]:
# fallback if yf fails: use SPX.csv as proxy for SPY
# spy = pd.read_csv('../../market_data/SPX.csv')
# spy.set_index("Date", inplace=True)

In [60]:
spy.index = pd.to_datetime(spy.index)
spy.sort_index(ascending=False, inplace=True)
spy.head()

# calculate daily log returns
spy['Log Return'] = np.log(spy['Close'] / spy['Close'].shift(-1))    # shift direction depends on sorting

In [61]:
spy.describe()

Price,Close,High,Low,Open,Volume,Log Return
count,792.000000,792.000000,792.000000,792.000000,7.920000e+02,791.000000
mean,528.734330,531.209670,525.742271,528.573991,7.116751e+07,0.000795
std,96.154148,96.436162,95.814931,96.273854,2.595686e+07,0.009523
min,364.684875,367.049580,363.194919,366.934233,2.604870e+07,-0.060327
25%,435.126213,436.874430,433.518648,434.818085,5.527650e+07,-0.003312
50%,534.867249,538.201837,532.120885,535.603988,6.870495e+07,0.000984
75%,596.079895,598.280806,593.764326,595.751346,8.437422e+07,0.005912
max,695.489990,697.840027,693.940002,697.049988,2.566114e+08,0.099863


In [62]:
# plot SPX Latest prices
fig = px.line(spy, y='Close', title='SPX Index')
fig.show()

# plot SPX Log Returns
fig = px.line(spy, y='Log Return', title='SPX Log Returns')
fig.show()

In [63]:
# fit lognormal distribution to log returns
log_returns = spy['Log Return']
mean = log_returns.mean()
std = log_returns.std()

# plot histogram of log returns and fitted lognormal distribution
fig = px.histogram(spy, x='Log Return', title='SPX Log Returns Histogram', nbins=100)
fig.show()

# plot the pdf of the lognormal distribution
x = np.linspace(log_returns.min(), log_returns.max(), 100)
pdf = 1/(std*np.sqrt(2*np.pi))*np.exp(-(x-mean)**2/(2*std**2))
px.line(x=x, y=pdf)

In [64]:
# calculate skewness of log returns as moment of order 3
log_returns = spy['Log Return']
mean = log_returns.mean()
std = log_returns.std()
skewness = ((log_returns - mean)**3).mean() / std**3
print(f"Mean of SPX Log Returns: {mean}")
print(f"Skewness of SPX Log Returns: {skewness}")

# calculate excess kurtosis of log returns as moment of order 4 minus 3 (actual formula)
kurtosis = ((log_returns - mean)**4).mean() / std**4
excess_kurtosis = kurtosis - 3
print(f"Excess Kurtosis of SPX Log Returns: {excess_kurtosis}")

Mean of SPX Log Returns: 0.0007947019506719599
Skewness of SPX Log Returns: 0.587187979659902
Excess Kurtosis of SPX Log Returns: 17.914452686347897


In [65]:


# Load SPY daily datar = spy["Close"].pct_change().dropna()          # daily log-approx returns

# ── Moments ─────────────────────────────────────────────────────────
mu    = log_returns.mean()                                # daily mean
sigma = log_returns.std()                                 # daily std dev
skew  = log_returns.skew()                                # skewness
kurt  = log_returns.kurtosis()                            # excess kurtosis

print(f"Mean (daily):     {mu:.4%}")
print(f"Volatility:       {sigma:.4%}  (annualized: {sigma*np.sqrt(252):.2%})")
print(f"Skewness:         {skew:.3f}   (normal = 0)")
print(f"Excess Kurtosis:  {kurt:.3f}   (normal = 0)")


Mean (daily):     0.0795%
Volatility:       0.9523%  (annualized: 15.12%)
Skewness:         0.589   (normal = 0)
Excess Kurtosis:  18.089   (normal = 0)
